Accelerator: GPU T4 x2, Internet on, and attach the `kaggle_prepare_data` output under Add Input → Your Work → Notebook Output.

In [ ]:
REPO_URL = "https://github.com/<you>/candidate_reranker.git"
BRANCH = "main"

In [ ]:
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/candidate_reranker")
if not (CODE / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)],
                   check=True)

sys.path.insert(0, str(CODE / "src"))
import kaggle_env as K

COMMIT = K.sync(REPO_URL, BRANCH)     # rerun this cell after every push
K.gpu_info()
env = K.prepare(COMMIT)

In [ ]:
assert K.run(env, "selftest.py",
             "--work", "/kaggle/working",
             "--n_utts", "3",
             "--n_candidates", "5") == 0

Dump candidates. `test-clean` to report on, `dev-*` to tune on.

In [ ]:
def dump(split, **kw):
    tag = f"{split}-{COMMIT}"
    args = ["--source", "librispeech",
            "--path", env.librispeech / split,
            "--base_model", env.base_model,
            "--adapter", env.adapter,
            "--out", env.results / f"{tag}.jsonl",
            "--tag", tag, "--resume"]
    for k, v in kw.items():
        args += [f"--{k}", v]
    return K.run(env, "dump_candidates.py", *args)

for split in ["dev-clean", "dev-other", "test-clean"]:
    dump(split, n_candidates=15, n_steps=4)

Standing measurement: oracle gap, coverage, diversity.

In [ ]:
K.run(env, "analyze.py",
      env.results / f"test-clean-{COMMIT}.jsonl",
      "--json", env.results / f"test-clean-{COMMIT}.stats.json")

Phase A. Tune on dev, then report composition and voting on test.

In [ ]:
K.run(env, "tune.py",
      "--dev", env.results / f"dev-clean-{COMMIT}.jsonl",
             env.results / f"dev-other-{COMMIT}.jsonl",
      "--test", env.results / f"test-clean-{COMMIT}.jsonl",
      "--json", env.results / f"tune-{COMMIT}.json")

In [ ]:
K.run(env, "analyze_compose.py",
      env.results / f"test-clean-{COMMIT}.jsonl",
      "--alpha", 1.0,
      "--json", env.results / f"compose-{COMMIT}.json")